In [1]:
from visualisation_donnees import charger_routes, charger_trips, charger_parcours

df_routes = charger_routes("data/idfm")
df_routes.head(20)

,route_id,agency_id,route_short_name,route_type,route_color,route_text_color,mode
456,IDFM:C01371,IDFM:Operator_100,1,1,FFBE00,000000,metro
457,IDFM:C01372,IDFM:Operator_100,2,1,0055C8,FFFFFF,metro
458,IDFM:C01373,IDFM:Operator_100,3,1,6E6E00,FFFFFF,metro
462,IDFM:C01384,IDFM:Operator_100,14,1,640082,FFFFFF,metro
463,IDFM:C01386,IDFM:Operator_100,3B,1,82C8E6,000000,metro
464,IDFM:C01387,IDFM:Operator_100,7B,1,82DC73,000000,metro
475,IDFM:C01377,IDFM:Operator_100,7,1,FF82B4,000000,metro
1234,IDFM:C01383,IDFM:Operator_100,13,1,82C8E6,000000,metro
1355,IDFM:C01374,IDFM:Operator_100,4,1,A0006E,FFFFFF,metro
1356,IDFM:C01375,IDFM:Operator_100,5,1,FF5A00,000000,metro


In [2]:
df_trips = charger_trips("data/idfm", df_routes)
df_trips.head()

,route_id,trip_id,shape_id,agency_id,route_short_name,route_type,route_color,route_text_color,mode
0,IDFM:C01381,IDFM:RATP:COU_RATP_11DIGAILLAC_2026_Chat01Robp...,IDFM:shp_1_41,IDFM:Operator_100,11,1,6E491E,FFFFFF,metro
1,IDFM:C01381,IDFM:RATP:COU_RATP_11DIGAILLAC_2026_Robp02Robp...,IDFM:shp_1_41,IDFM:Operator_100,11,1,6E491E,FFFFFF,metro
2,IDFM:C01381,IDFM:RATP:COU_RATP_11DIGAILLAC_2026_PteL01Chat...,IDFM:shp_1_119,IDFM:Operator_100,11,1,6E491E,FFFFFF,metro
3,IDFM:C01381,IDFM:RATP:COU_RATP_11DIGAILLAC_2026_Robp02Robp...,IDFM:shp_1_222,IDFM:Operator_100,11,1,6E491E,FFFFFF,metro
4,IDFM:C01381,IDFM:RATP:COU_RATP_11DIGAILLAC_2026_Robp01Robp...,IDFM:shp_1_220,IDFM:Operator_100,11,1,6E491E,FFFFFF,metro


In [3]:
df_parcours = charger_parcours("data/idfm", df_trips)
df_parcours.head()

,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,route_id,agency_id,route_short_name,route_type,route_color,route_text_color,mode
16670,IDFM:shp_1_100,48.848721,2.298968,0,IDFM:C01376,IDFM:Operator_100,6,1,82DC73,000000,metro
16671,IDFM:shp_1_100,48.847824,2.301867,1,IDFM:C01376,IDFM:Operator_100,6,1,82DC73,000000,metro
16672,IDFM:shp_1_100,48.847511,2.302942,2,IDFM:C01376,IDFM:Operator_100,6,1,82DC73,000000,metro
16673,IDFM:shp_1_100,48.847511,2.302942,3,IDFM:C01376,IDFM:Operator_100,6,1,82DC73,000000,metro
16674,IDFM:shp_1_100,48.846603,2.306028,4,IDFM:C01376,IDFM:Operator_100,6,1,82DC73,000000,metro


In [4]:
import geopandas as gpd
import pandas as pd
from visualisation_donnees import (
    charger_routes, charger_trips, charger_parcours,
    construire_lignes_idfm, charger_arrets, charger_stations,
)

df_routes = charger_routes("data/idfm")
df_trips = charger_trips("data/idfm", df_routes)
df_parcours = charger_parcours("data/idfm", df_trips)
gpd_idfm = construire_lignes_idfm(df_parcours)
df_arrets = charger_arrets("data/idfm", df_trips)
df_stations = charger_stations("data/idfm", df_arrets)

In [5]:
gpd_stations = gpd.GeoDataFrame(
    df_stations,
    geometry=gpd.points_from_xy(df_stations["stop_lon"], df_stations["stop_lat"]),
    crs="EPSG:4326",
).to_crs("EPSG:2154")

In [6]:
gpd_idfm["longueur"] = gpd_idfm.length
trace_ref = gpd_idfm.sort_values("longueur", ascending=False).drop_duplicates("route_id")
trace_ref = pd.DataFrame(trace_ref[["route_id", "geometry"]]).rename(columns={"geometry": "trace"})

In [7]:
stations = gpd_stations.merge(trace_ref, on="route_id")

# distance entre la station et le tracé (pour repérer les stations d'une autre branche)
stations["distance_trace"] = stations.apply(lambda r: r["trace"].distance(r["geometry"]), axis=1)
# position le long du tracé, en km
stations["position_km"] = stations.apply(lambda r: r["trace"].project(r["geometry"]) / 1000, axis=1)

stations = stations[stations["distance_trace"] < 300]
stations = stations.sort_values(["route_short_name", "position_km"])

In [11]:
# faire commencer chaque ligne à 0 km
stations["position_km"] = stations["position_km"] - stations.groupby("route_short_name")["position_km"].transform("min")

# arrondir à 1 chiffre après la virgule
stations["position_km"] = stations["position_km"].round(1)

In [12]:
stations.loc[stations["route_short_name"] == "14", ["stop_name", "position_km"]]

,stop_name,position_km
643,Saint-Denis - Pleyel,0.0
207,Mairie de Saint-Ouen,1.0
210,Saint-Ouen,2.4
150,Porte de Clichy,4.0
374,Pont Cardinet,4.7
407,Gare Saint-Lazare,6.5
438,Madeleine,7.3
271,Pyramides,8.1
490,Châtelet,9.5
325,Gare de Lyon,12.0
